# Day 3: Data Integration & Mini Project
## Stage 1: Fundamental Data Science & Analytics — Epson Training

**Learning Objectives:**
- Perform data wrangling on MES, ERP, and IoT data sources
- Conduct a structured Exploratory Data Analysis (EDA)
- Produce an Analysis Report with visualizations relevant to Epson operations

> 💡 **Google Colab Tip:** This notebook produces a complete EDA report. Run all cells in order.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

sns.set_theme(style="whitegrid", palette="muted")
np.random.seed(42)

print("✅ Libraries loaded")

---
## Part 1: Simulate Multi-Source Data (MES / ERP / IoT)

In real Epson operations, data comes from:
- **MES (Manufacturing Execution System):** Work orders, production counts, machine state
- **ERP (Enterprise Resource Planning):** Materials, orders, inventory, costs
- **IoT Sensors:** Real-time temperature, vibration, pressure from machines

In [ ]:
# ---- IoT Sensor Data ----
n_iot = 500
start = datetime(2024, 1, 1)
machines = ["M001", "M002", "M003", "M004", "M005"]

iot_df = pd.DataFrame({
    "sensor_time": [start + timedelta(hours=i * 0.5) for i in range(n_iot)],
    "machine_id": np.random.choice(machines, n_iot),
    "temperature_c": np.round(np.random.normal(72, 3, n_iot), 2),
    "vibration_mms": np.round(np.abs(np.random.normal(3, 1.2, n_iot)), 3),
    "pressure_hpa": np.round(np.random.normal(1013, 5, n_iot), 1),
    "current_amp": np.round(np.random.normal(12.5, 1.0, n_iot), 2),
})

# Inject some anomalies
anomaly_idx = np.random.choice(n_iot, 20, replace=False)
iot_df.loc[anomaly_idx, "temperature_c"] = np.random.uniform(88, 98, 20)
iot_df.loc[anomaly_idx[:5], "vibration_mms"] = np.random.uniform(8, 12, 5)

print(f"IoT data: {iot_df.shape}")
iot_df.head()

In [ ]:
# ---- MES (Manufacturing Execution System) Data ----
n_mes = 300
product_types = ["Printer_A4", "Printer_A3", "Scanner_Pro", "Projector_HD"]
shifts = ["Day", "Night"]

mes_df = pd.DataFrame({
    "work_order_id": [f"WO-{2024000 + i}" for i in range(n_mes)],
    "machine_id": np.random.choice(machines, n_mes),
    "product_type": np.random.choice(product_types, n_mes),
    "shift": np.random.choice(shifts, n_mes),
    "planned_qty": np.random.randint(100, 300, n_mes),
    "actual_qty": np.random.randint(90, 305, n_mes),
    "defect_qty": np.random.randint(0, 15, n_mes),
    "downtime_min": np.where(np.random.rand(n_mes) < 0.15,
                             np.random.randint(5, 120, n_mes), 0),
    "production_date": [start + timedelta(days=i // 6) for i in range(n_mes)],
})

mes_df["efficiency"] = np.round(mes_df["actual_qty"] / mes_df["planned_qty"] * 100, 2)
mes_df["defect_rate"] = np.round(mes_df["defect_qty"] / mes_df["actual_qty"] * 100, 2)

print(f"MES data: {mes_df.shape}")
mes_df.head()

In [ ]:
# ---- ERP Data ----
n_erp = 150
erp_df = pd.DataFrame({
    "order_id": [f"ORD-{10000 + i}" for i in range(n_erp)],
    "product_type": np.random.choice(product_types, n_erp),
    "customer_region": np.random.choice(["Asia", "Europe", "Americas", "Japan"], n_erp),
    "order_qty": np.random.randint(50, 500, n_erp),
    "unit_cost_usd": np.random.choice([120.0, 180.0, 95.0, 250.0], n_erp),
    "material_available": np.random.choice([True, True, True, False], n_erp),
    "order_date": [start + timedelta(days=i // 3) for i in range(n_erp)],
    "target_delivery_days": np.random.randint(5, 20, n_erp),
})

erp_df["order_value_usd"] = erp_df["order_qty"] * erp_df["unit_cost_usd"]

print(f"ERP data: {erp_df.shape}")
erp_df.head()

---
## Part 2: Data Wrangling

In [ ]:
# --- Missing Value Analysis ---

# Introduce missing values to simulate real-world data quality issues
iot_dirty = iot_df.copy()
missing_idx = np.random.choice(n_iot, size=int(n_iot * 0.08), replace=False)
iot_dirty.loc[missing_idx[:20], "temperature_c"] = np.nan
iot_dirty.loc[missing_idx[20:30], "vibration_mms"] = np.nan
iot_dirty.loc[missing_idx[30:35], "pressure_hpa"] = np.nan

print("=== Missing Values in IoT Data ===")
missing_summary = iot_dirty.isnull().sum()
missing_pct = (iot_dirty.isnull().sum() / len(iot_dirty) * 100).round(2)
print(pd.DataFrame({"Missing Count": missing_summary, "Missing %": missing_pct}))

# Strategy 1: Fill temperature with machine-specific median
iot_dirty["temperature_c"] = iot_dirty.groupby("machine_id")["temperature_c"].transform(
    lambda x: x.fillna(x.median())
)

# Strategy 2: Forward-fill vibration (carry last known value)
iot_dirty["vibration_mms"] = iot_dirty["vibration_mms"].fillna(method="ffill")

# Strategy 3: Fill pressure with global median
iot_dirty["pressure_hpa"] = iot_dirty["pressure_hpa"].fillna(iot_dirty["pressure_hpa"].median())

print("\nMissing values after cleaning:")
print(iot_dirty.isnull().sum())

iot_clean = iot_dirty.copy()

In [ ]:
# --- Duplicate Detection & Removal ---
# Introduce duplicates
iot_with_dups = pd.concat([iot_clean, iot_clean.sample(10)], ignore_index=True)
print(f"Rows before dedup: {len(iot_with_dups)}")
iot_with_dups = iot_with_dups.drop_duplicates()
print(f"Rows after dedup:  {len(iot_with_dups)}")

# --- Data Type Normalization ---
mes_df["production_date"] = pd.to_datetime(mes_df["production_date"])
erp_df["order_date"] = pd.to_datetime(erp_df["order_date"])

# Extract useful time features
mes_df["production_month"] = mes_df["production_date"].dt.month
mes_df["production_week"] = mes_df["production_date"].dt.isocalendar().week.astype(int)
mes_df["day_of_week"] = mes_df["production_date"].dt.day_name()

print("\nMES dtypes after cleaning:")
print(mes_df.dtypes)

In [ ]:
# --- Outlier Detection (IQR method) ---

def detect_outliers_iqr(series, name="variable"):
    """Detect outliers using the IQR method."""
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = series[(series < lower) | (series > upper)]
    print(f"  {name}: Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}")
    print(f"    Bounds: [{lower:.2f}, {upper:.2f}]")
    print(f"    Outliers: {len(outliers)} ({len(outliers)/len(series)*100:.1f}%)")
    return lower, upper

print("=== Outlier Analysis ===")
bounds = {}
for col in ["temperature_c", "vibration_mms", "pressure_hpa"]:
    lo, hi = detect_outliers_iqr(iot_clean[col], col)
    bounds[col] = (lo, hi)

# Cap outliers at bounds (Winsorization)
for col, (lo, hi) in bounds.items():
    iot_clean[col] = iot_clean[col].clip(lower=lo, upper=hi)

print("\n✅ Outliers capped using Winsorization")

---
## Part 3: Exploratory Data Analysis (EDA)

In [ ]:
# --- Univariate Analysis ---
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Univariate EDA — IoT & MES Data", fontsize=13, fontweight="bold")

# IoT sensor distributions
iot_cols = [("temperature_c", "Temperature (°C)", "tomato"),
            ("vibration_mms", "Vibration (mm/s)", "steelblue"),
            ("pressure_hpa", "Pressure (hPa)", "seagreen")]

for ax, (col, label, color) in zip(axes[0], iot_cols):
    ax.hist(iot_clean[col], bins=30, color=color, edgecolor="white", alpha=0.8)
    ax.axvline(iot_clean[col].mean(), color="black", linestyle="--", linewidth=1.2,
               label=f"Mean: {iot_clean[col].mean():.1f}")
    ax.set_title(f"Distribution of {label}")
    ax.set_xlabel(label)
    ax.set_ylabel("Frequency")
    ax.legend()

# MES distributions
mes_cols = [("efficiency", "Production Efficiency (%)", "#8172B2"),
            ("defect_rate", "Defect Rate (%)", "#C44E52"),
            ("downtime_min", "Downtime (min)", "#CCB974")]

for ax, (col, label, color) in zip(axes[1], mes_cols):
    ax.hist(mes_df[col], bins=25, color=color, edgecolor="white", alpha=0.8)
    ax.axvline(mes_df[col].mean(), color="black", linestyle="--", linewidth=1.2,
               label=f"Mean: {mes_df[col].mean():.1f}")
    ax.set_title(f"Distribution of {label}")
    ax.set_xlabel(label)
    ax.set_ylabel("Frequency")
    ax.legend()

plt.tight_layout()
plt.savefig("day3_univariate_eda.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
# --- Bivariate Analysis ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Bivariate EDA", fontsize=13, fontweight="bold")

# Machine efficiency by product type
sns.boxplot(data=mes_df, x="product_type", y="efficiency",
            ax=axes[0], palette="Set2")
axes[0].set_title("Efficiency by Product Type")
axes[0].set_xlabel("Product")
axes[0].tick_params(axis="x", rotation=20)

# Defect rate by shift and machine
shift_machine = mes_df.groupby(["shift", "machine_id"])["defect_rate"].mean().unstack()
shift_machine.plot(kind="bar", ax=axes[1], rot=0)
axes[1].set_title("Avg Defect Rate: Shift × Machine")
axes[1].set_ylabel("Defect Rate (%)")
axes[1].legend(title="Machine", bbox_to_anchor=(1.01, 1))

# IoT: temperature vs vibration
sample = iot_clean.sample(200)
sc = axes[2].scatter(sample["temperature_c"], sample["vibration_mms"],
                     c=sample["current_amp"], cmap="plasma", alpha=0.6, s=25)
plt.colorbar(sc, ax=axes[2], label="Current (A)")
axes[2].set_title("Temperature vs Vibration\n(colored by Current)")
axes[2].set_xlabel("Temperature (°C)")
axes[2].set_ylabel("Vibration (mm/s)")

plt.tight_layout()
plt.savefig("day3_bivariate_eda.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
# --- Correlation Analysis ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# IoT correlation heatmap
iot_corr = iot_clean[["temperature_c", "vibration_mms", "pressure_hpa", "current_amp"]].corr()
sns.heatmap(iot_corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, ax=axes[0])
axes[0].set_title("IoT Sensor Correlations")

# MES correlation heatmap
mes_corr = mes_df[["planned_qty", "actual_qty", "defect_qty", "downtime_min",
                   "efficiency", "defect_rate"]].corr()
sns.heatmap(mes_corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, ax=axes[1])
axes[1].set_title("MES Production Variable Correlations")

plt.tight_layout()
plt.savefig("day3_correlation_heatmaps.png", dpi=100, bbox_inches="tight")
plt.show()

---
## Part 4: Mini Project Output — EDA Analysis Report

In [ ]:
# ============================================================
#  EDA ANALYSIS REPORT: Epson Operations — January 2024
# ============================================================

print("=" * 62)
print("  EDA ANALYSIS REPORT: Epson Operations — January 2024")
print("=" * 62)

# Section 1: Data Overview
print("\n1. DATA OVERVIEW")
print(f"   IoT Sensor Records:    {len(iot_clean):>6,}")
print(f"   MES Work Orders:       {len(mes_df):>6,}")
print(f"   ERP Orders:            {len(erp_df):>6,}")
print(f"   Machines Monitored:    {iot_clean['machine_id'].nunique():>6}")
print(f"   Products Manufactured: {mes_df['product_type'].nunique():>6}")

# Section 2: Production Performance
print("\n2. PRODUCTION PERFORMANCE")
overall_efficiency = mes_df["efficiency"].mean()
print(f"   Overall avg efficiency:  {overall_efficiency:.1f}%")
best_product = mes_df.groupby("product_type")["efficiency"].mean().idxmax()
best_eff = mes_df.groupby("product_type")["efficiency"].mean().max()
print(f"   Best product efficiency: {best_product} ({best_eff:.1f}%)")
total_downtime = mes_df["downtime_min"].sum()
downtime_pct = (mes_df["downtime_min"] > 0).mean() * 100
print(f"   Total downtime:          {total_downtime:,} min ({downtime_pct:.1f}% of records)")

# Section 3: Quality
print("\n3. QUALITY ANALYSIS")
overall_defect = mes_df["defect_rate"].mean()
worst_machine_def = mes_df.groupby("machine_id")["defect_rate"].mean().idxmax()
best_machine_def = mes_df.groupby("machine_id")["defect_rate"].mean().idxmin()
print(f"   Overall avg defect rate: {overall_defect:.2f}%")
print(f"   Highest defect machine:  {worst_machine_def}")
print(f"   Lowest defect machine:   {best_machine_def}")

# Section 4: IoT Sensor Insights
print("\n4. IoT SENSOR INSIGHTS")
overheat_pct = (iot_clean["temperature_c"] > 80).mean() * 100
high_vib_pct = (iot_clean["vibration_mms"] > 6).mean() * 100
print(f"   High temp events (>80°C):     {overheat_pct:.1f}%")
print(f"   High vibration events (>6):   {high_vib_pct:.1f}%")
print(f"   Avg temperature:              {iot_clean['temperature_c'].mean():.2f}°C")
print(f"   Avg vibration:                {iot_clean['vibration_mms'].mean():.3f} mm/s")

# Section 5: ERP Business
print("\n5. ERP BUSINESS METRICS")
total_order_value = erp_df["order_value_usd"].sum()
material_gap = (~erp_df["material_available"]).sum()
top_region = erp_df.groupby("customer_region")["order_value_usd"].sum().idxmax()
print(f"   Total order value:  ${total_order_value:>12,.0f}")
print(f"   Orders w/o material: {material_gap} ({material_gap/len(erp_df)*100:.1f}%)")
print(f"   Top revenue region: {top_region}")

print("\n6. KEY RECOMMENDATIONS")
print(f"   ⚠️  Investigate {worst_machine_def}: highest defect rate")
print(f"   🔥 {overheat_pct:.0f}% of sensor readings show high temperature — review cooling")
print(f"   📦 {material_gap} orders at risk due to material unavailability")
print(f"   🏆 Replicate {best_product} production practices across all lines")
print("\n" + "=" * 62)

In [ ]:
# Final EDA Dashboard
fig = plt.figure(figsize=(18, 14))
fig.suptitle("Epson Operations EDA Report — January 2024",
             fontsize=15, fontweight="bold", y=1.01)

gs = fig.add_gridspec(3, 3, hspace=0.45, wspace=0.35)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[0, 2])
ax4 = fig.add_subplot(gs[1, 0])
ax5 = fig.add_subplot(gs[1, 1])
ax6 = fig.add_subplot(gs[1, 2])
ax7 = fig.add_subplot(gs[2, 0:2])
ax8 = fig.add_subplot(gs[2, 2])

# 1. IoT temperature distribution
ax1.hist(iot_clean["temperature_c"], bins=25, color="tomato", edgecolor="white", alpha=0.8)
ax1.set_title("IoT: Temperature Distribution")
ax1.set_xlabel("°C")

# 2. MES efficiency by machine
eff_by_m = mes_df.groupby("machine_id")["efficiency"].mean()
ax2.bar(eff_by_m.index, eff_by_m.values, color="#4C72B0", edgecolor="white")
ax2.axhline(100, color="gray", linestyle="--", linewidth=0.8)
ax2.set_title("MES: Avg Efficiency by Machine")
ax2.set_ylabel("%")

# 3. ERP revenue by region
rev_by_region = erp_df.groupby("customer_region")["order_value_usd"].sum() / 1e6
ax3.pie(rev_by_region.values, labels=rev_by_region.index,
        autopct="%1.1f%%", colors=sns.color_palette("muted", 4))
ax3.set_title("ERP: Revenue by Region")

# 4. Defect rate by machine (MES)
sns.boxplot(data=mes_df, x="machine_id", y="defect_rate", ax=ax4, palette="Reds")
ax4.set_title("MES: Defect Rate by Machine")
ax4.set_ylabel("%")

# 5. IoT vibration by machine
sns.violinplot(data=iot_clean, x="machine_id", y="vibration_mms",
               ax=ax5, palette="Blues", inner="quartile")
ax5.set_title("IoT: Vibration by Machine")
ax5.set_ylabel("mm/s")

# 6. Downtime distribution
downtime_nonzero = mes_df[mes_df["downtime_min"] > 0]["downtime_min"]
ax6.hist(downtime_nonzero, bins=20, color="#CCB974", edgecolor="white")
ax6.set_title(f"MES: Downtime Distribution\n(n={len(downtime_nonzero)} events)")
ax6.set_xlabel("Minutes")

# 7. Time-series: daily defect rate (MES)
daily_defect = mes_df.groupby("production_date")["defect_rate"].mean()
ax7.plot(daily_defect.index, daily_defect.values, color="#C44E52", linewidth=1.5)
ax7.fill_between(daily_defect.index, daily_defect.values, alpha=0.2, color="#C44E52")
ax7.axhline(daily_defect.mean(), color="black", linestyle="--",
            label=f"Mean: {daily_defect.mean():.2f}%")
ax7.set_title("Daily Average Defect Rate")
ax7.set_ylabel("%")
ax7.legend()
ax7.tick_params(axis="x", rotation=20)

# 8. ERP product order distribution
prod_counts = erp_df["product_type"].value_counts()
ax8.barh(prod_counts.index, prod_counts.values,
         color=sns.color_palette("muted", len(prod_counts)))
ax8.set_title("ERP: Orders by Product")
ax8.set_xlabel("Count")

plt.savefig("day3_eda_report_dashboard.png", dpi=100, bbox_inches="tight")
plt.show()
print("\n✅ EDA Report Dashboard saved as day3_eda_report_dashboard.png")

---
## ✅ Day 3 Mini Project Checklist

Complete the following as your Day 3 deliverable:

1. **Data Collection**: Load data from at least two sources (IoT + MES or IoT + ERP).
2. **Data Cleaning**: Handle missing values and remove duplicates. Document your strategy.
3. **Outlier Analysis**: Identify and treat outliers in sensor data.
4. **Univariate EDA**: Create histograms or KDE plots for at least 4 variables.
5. **Bivariate EDA**: Create at least 3 bivariate plots (scatter, box, heatmap).
6. **Business Summary**: Write a 5-bullet summary of your key findings relevant to Epson operations.

Save your final notebook and export the dashboard chart as a PNG for your report.